In [ ]:
import sys
from pathlib import Path
# Add parent directory to path to import synctools modules
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import matplotlib.pyplot as plt
from speckit import noise

from synctools.synchronization import sync_signals

# Configure SpecKit parameters for spectral analysis
p_lpsd = {
    "olap": "default",
    "bmin": 1.0,
    "Lmin": 1,
    "Jdes": 500,
    "Kdes": 100,
    "order": 1,
    "win": np.kaiser,
    "psll": 250,
}

# Set up matplotlib for better plots
plt.style.use('default')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['font.size'] = 10

# 2-Signal Synchronization Demonstration

This notebook demonstrates the `sync_signals` function for **2-signal synchronization**. 

**Both TwoSignals and ThreeSignals represent null measurements** that should cancel to zero when the participating signals are properly synchronized. The `sync_signals` function optimizes for the time offsets that, when applied to the input signals, recover the null measurement.

For 2-signal sync, we use two signals that measure the same quantity (e.g., the same laser beat frequency from different phasemeters). When synchronized, their combination (y1 - y2) should be zero. When unsynchronized, time offsets cause the null measurement to fail, showing large artifacts. The synchronization process recovers the correct offsets to restore the null.

## Setup: Generate Synthetic Signals with Known Time Offset

We'll create two frequency signals that measure the same beat frequency, with an intentional time offset to demonstrate the synchronization capability.

In [ ]:
# Signal generation parameters
N = int(1e4)  # Number of samples
noise_level = 1000  # Laser frequency noise level
nu_laser = 299792458 / 1064.5e-9  # Nominal laser frequency (Hz)
fs = 3.3  # Sampling frequency of the phasemeters (Hz)

# Generate random walk noise (red noise) for laser frequency noise
rw1 = noise.red_noise(fs, f_min=1e-3, seed=41)
rw2 = noise.red_noise(fs, f_min=1e-3, seed=42)

# Create laser frequencies with different offsets
laser1 = nu_laser - 5e6 + noise_level * rw1.get_series(N)  # Laser frequency (Hz)
laser2 = nu_laser - 15e6 + noise_level * rw2.get_series(N)  # Laser frequency (Hz)

# Create phasemeter signals (PIR values)
# For 2-signal null measurement: both measure the same beat frequency
# When synchronized, y1 - y2 should be zero (null measurement)
y1 = laser1 - laser2  # Phasemeter 1, PIR value (Hz)
y2 = laser1 - laser2  # Phasemeter 2, same measurement (Hz)

print(f"Signal means: y1={np.mean(y1)/1e6:.2f} MHz, y2={np.mean(y2)/1e6:.2f} MHz")
print(f"Signal lengths: {len(y1)} samples")
print(f"Sampling rate: {fs} Hz")
print(f"Duration: {len(y1)/fs:.1f} seconds")
print(f"\nWhen perfectly synchronized, y1 - y2 should be zero (null measurement)")
print(f"Initial difference (y1 - y2) RMS: {np.std(y1 - y2):.2e} Hz")

## Plot 1: Input Signals

Visualize the two input frequency signals.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

time_axis = np.arange(len(y1)) / fs

axes[0].plot(time_axis, y1 / 1e6, color='#1f77b4', linewidth=1.5, label='Signal 1 (y1)')
axes[0].set_ylabel('Frequency (MHz)', fontsize=11)
axes[0].set_title('Input Frequency Signals', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper right', edgecolor='black', fancybox=True, shadow=True, framealpha=1)
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_axis, y2 / 1e6, color='#ff7f0e', linewidth=1.5, label='Signal 2 (y2)')
axes[1].set_xlabel('Time (s)', fontsize=11)
axes[1].set_ylabel('Frequency (MHz)', fontsize=11)
axes[1].legend(loc='upper right', edgecolor='black', fancybox=True, shadow=True, framealpha=1)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Introduce Time Offset

Now we'll introduce an intentional time offset to simulate unsynchronized phasemeters. This will cause the null measurement to fail, showing large artifacts. Synchronization will recover the null by finding the correct offsets.

In [ ]:
# Introduce time offset (in samples)
shift = int(1.2 * fs)  # ~5 seconds offset for signal 2

# Apply shift (signal 1 is the reference, signal 2 is shifted)
y1_shifted = y1[:-2*shift]
y2_shifted = y2[shift:-shift] if shift <= len(y2) - shift else y2[shift:]

# Find common length
min_len = min(len(y1_shifted), len(y2_shifted))
y1_shifted = y1_shifted[:min_len]
y2_shifted = y2_shifted[:min_len]

# True offset in seconds
true_offset = shift / fs

print(f"Applied time offset:")
print(f"  Signal 2: {true_offset:.3f} seconds ({shift} samples)")
print(f"Final signal length: {len(y1_shifted)} samples ({len(y1_shifted)/fs:.1f} seconds)")

# Verify both signals have the same length
assert len(y1_shifted) == len(y2_shifted)

## Plot 2: Null Measurement Before Synchronization

The 2-signal combination is the null measurement y1 - y2. When perfectly synchronized, this should be zero. Before synchronization, the time offset causes the null measurement to fail, showing large artifacts.

In [ ]:
# Compute unsynchronized difference
unsynced_difference = y1_shifted - y2_shifted

fig, ax = plt.subplots(figsize=(10, 4))

time_axis_shifted = np.arange(len(unsynced_difference)) / fs

ax.plot(time_axis_shifted, unsynced_difference, color='#d62728', linewidth=1, alpha=0.7)
ax.set_xlabel('Time (s)', fontsize=11)
ax.set_ylabel('Frequency (Hz)', fontsize=11)
ax.set_title('Unsynchronized 2-Signal Null Measurement (y1 - y2) Before Synchronization', 
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f"Unsynchronized null measurement RMS: {np.std(unsynced_difference):.2e} Hz")
print(f"Unsynchronized null measurement mean: {np.mean(unsynced_difference):.2e} Hz")
print(f"Note: When synchronized, this should be close to zero (null measurement)")

## Synchronize the Signals

Now we'll use the `sync_signals` function to automatically detect and correct the time offset. The optimization finds the offsets that minimize the null measurement residual, recovering the null (y1 - y2 ≈ 0).

In [ ]:
# Prepare input signals
in_signals = [y1_shifted, y2_shifted]

# Initial offset guess (can be None for automatic detection)
init_offsets = [0.9]  # Will default to [0.0]

# Call sync_signals
print("Starting 2-signal synchronization...")
print("=" * 60)
unsynced_obj, synced_obj = sync_signals(
    in_signals=in_signals,
    fs=fs,
    p_lpsd=p_lpsd,
    init_offsets=init_offsets,
    model="fluc",  # Use fluctuation frequency model
    domain="freq",  # Optimize in time domain
    method="Nelder-Mead",  # Optimization method
    interp_order=121,  # High-order interpolation for accuracy
    n_truncate=None,  # Auto-calculate truncation
    clock_refs=None,  # No clock jitter correction
)

print("=" * 60)
print("\nSynchronization complete!")
print(f"\nOptimized time offset:")
print(f"  Signal 2: {synced_obj.timer_offsets[0]:.6f} seconds (true: {true_offset:.6f} s)")
print(f"\nOffset error: {abs(synced_obj.timer_offsets[0] + true_offset):.6f} seconds")
print(f"\nTDIR precision: {synced_obj.TDIR_precision:.2e} seconds")

## Plot 3: Synchronized vs Unsynchronized Null Measurement (Time Domain)

Compare the synchronized and unsynchronized null measurements in the time domain. After synchronization, the null measurement should be close to zero.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Unsynchronized difference
time_unsynced = unsynced_obj.tau
axes[0].plot(time_unsynced, unsynced_obj.main.freq, color='#d62728', linewidth=1, alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=0.5, alpha=0.5)
axes[0].set_ylabel('Frequency (Hz)', fontsize=11)
axes[0].set_title('Unsynchronized Null Measurement (y1 - y2)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].text(0.02, 0.95, f'RMS: {np.std(unsynced_obj.main.freq):.2e} Hz', 
             transform=axes[0].transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Synchronized difference
time_synced = synced_obj.tau
axes[1].plot(time_synced, synced_obj.freq['time'], color='#2ca02c', linewidth=1.5)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=0.5, alpha=0.5, label='Null (zero)')
axes[1].set_xlabel('Time (s)', fontsize=11)
axes[1].set_ylabel('Frequency (Hz)', fontsize=11)
axes[1].set_title('Synchronized Null Measurement (y1 - y2)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].text(0.02, 0.95, f'RMS: {np.std(synced_obj.freq["time"]):.2e} Hz', 
             transform=axes[1].transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

fig.tight_layout()
plt.show()

print(f"Improvement factor: {np.std(unsynced_obj.main.freq) / np.std(synced_obj.freq['time']):.2f}x reduction in RMS")

## Plot 4: Frequency Amplitude Spectral Density (ASD) Comparison

Compare the frequency noise spectra before and after synchronization.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Plot frequency ASDs
ax.loglog(unsynced_obj.main.fourier_freq, unsynced_obj.main.freq_asd, 
          color='#d62728', linewidth=2, label='Unsynchronized', alpha=0.8)
ax.loglog(synced_obj.fourier_freq, synced_obj.freq['asd'], 
          color='#2ca02c', linewidth=2, label='Synchronized', alpha=0.8)

ax.set_xlabel('Frequency (Hz)', fontsize=11)
ax.set_ylabel('Frequency ASD (Hz/√Hz)', fontsize=11)
ax.set_title('Frequency Amplitude Spectral Density', fontsize=12, fontweight='bold')
ax.legend(loc='best', edgecolor='black', fancybox=True, shadow=True, framealpha=1, fontsize=10)
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim([min(unsynced_obj.main.fourier_freq[1], synced_obj.fourier_freq[1]), 
             max(unsynced_obj.main.fourier_freq[-1], synced_obj.fourier_freq[-1])])

fig.tight_layout()
plt.show()

## Plot 5: Phase Amplitude Spectral Density (ASD) Comparison

Compare the phase noise spectra before and after synchronization. This is particularly important for precision measurements.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Plot phase ASDs
ax.loglog(unsynced_obj.main.fourier_freq, unsynced_obj.main.phase_asd, 
          color='#d62728', linewidth=2, label='Unsynchronized', alpha=0.8)
ax.loglog(synced_obj.fourier_freq, synced_obj.phase['asd'], 
          color='#2ca02c', linewidth=2, label='Synchronized', alpha=0.8)

# Add TDIR residual if available
if len(synced_obj.TDIR_residual_asd) > 0:
    ax.loglog(synced_obj.fourier_freq, synced_obj.TDIR_residual_asd, 
              color='#9467bd', linewidth=1.5, linestyle='--', 
              label='TDIR Residual', alpha=0.7)

ax.set_xlabel('Frequency (Hz)', fontsize=11)
ax.set_ylabel('Phase ASD (rad/√Hz)', fontsize=11)
ax.set_title('Phase Amplitude Spectral Density', fontsize=12, fontweight='bold')
ax.legend(loc='best', edgecolor='black', fancybox=True, shadow=True, framealpha=1, fontsize=10)
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim([min(unsynced_obj.main.fourier_freq[1], synced_obj.fourier_freq[1]), 
             max(unsynced_obj.main.fourier_freq[-1], synced_obj.fourier_freq[-1])])

fig.tight_layout()
plt.show()

## Plot 6: Phase Time Series Comparison

Visualize the phase evolution in the time domain before and after synchronization.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

# Unsynchronized phase
axes[0].plot(unsynced_obj.tau, unsynced_obj.main.phase, color='#d62728', linewidth=1, alpha=0.7)
axes[0].set_ylabel('Phase (rad)', fontsize=11)
axes[0].set_title('Unsynchronized Phase', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].text(0.02, 0.95, f'RMS: {np.std(unsynced_obj.main.phase):.4f} rad', 
             transform=axes[0].transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Synchronized phase
axes[1].plot(synced_obj.tau, synced_obj.phase['time'], color='#2ca02c', linewidth=1.5)
axes[1].set_xlabel('Time (s)', fontsize=11)
axes[1].set_ylabel('Phase (rad)', fontsize=11)
axes[1].set_title('Synchronized Phase', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].text(0.02, 0.95, f'RMS: {np.std(synced_obj.phase["time"]):.4f} rad', 
             transform=axes[1].transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

fig.tight_layout()
plt.show()

## Summary

This notebook demonstrated the `sync_signals` function's ability for **2-signal synchronization**:

1. **Automatically detect time offsets** between two phasemeter signals
2. **Correct synchronization errors** through optimization
3. **Recover the null measurement** (y1 - y2 ≈ 0) by finding the correct time offsets
4. **Significantly reduce noise** in the null measurement residual
5. **Improve spectral characteristics** in both frequency and phase domains

The synchronization process successfully recovered the known time offset and restored the null measurement, as evidenced by the reduction in RMS values (approaching zero) and improved spectral density characteristics.

**Key concept:**
- Both **2-signal and 3-signal sync use null measurements** that should cancel to zero when properly synchronized
- The `sync_signals` function optimizes for time offsets that, when applied to the input signals, recover the null measurement
- When unsynchronized, time offsets cause the null measurement to fail, showing large artifacts
- After synchronization, the null measurement is recovered (close to zero, with only residual noise)